In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

data = pd.read_csv("data_grupp_13.csv")
X = data.drop(columns=["target"])
y = data["target"]

X_scaled = StandardScaler().fit_transform(X)

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_base = SVC(random_state=42)

svm_scores = cross_validate(
    svm_base,
    X_scaled,
    y,
    cv=cv5,
    scoring={"acc": "accuracy", "f1_macro": "f1_macro"},
)

print(
    f"SVM Accuracy:  {svm_scores['test_acc'].mean():.4f} +/-"
    f" {svm_scores['test_acc'].std():.4f}"
)
print(
    f"SVM F1 macro:  {svm_scores['test_f1_macro'].mean():.4f} +/-"
    f" {svm_scores['test_f1_macro'].std():.4f}"
)

SVM Accuracy:  0.9958 +/- 0.0083
SVM F1 macro:  0.9958 +/- 0.0083


In [4]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.svm import SVC

cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_base = SVC(random_state=42)

svm_scores = cross_validate(
    svm_base,
    X_scaled,
    y,
    cv=cv5,
    scoring={"acc": "accuracy", "f1_macro": "f1_macro"},
)

print(
    f"SVM Accuracy:  {svm_scores['test_acc'].mean():.4f} +/-"
    f" {svm_scores['test_acc'].std():.4f}"
)
print(
    f"SVM F1 macro:  {svm_scores['test_f1_macro'].mean():.4f} +/-"
    f" {svm_scores['test_f1_macro'].std():.4f}"
)

SVM Accuracy:  0.9958 +/- 0.0083
SVM F1 macro:  0.9958 +/- 0.0083


In [5]:
import time
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GridSearchCV, LeaveOneOut, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.svm import SVC
import pandas as pd

n_folds = 5  # Byt mot gruppens tilldelade n från tilldelning_canvas.csv
cv_assigned = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

param_grid = {
    "C": [0.1, 1, 10, 100],
    "gamma": ["scale", 0.01, 0.1, 1],
}

svm_grid = GridSearchCV(
    SVC(kernel="rbf", random_state=42),
    param_grid=param_grid,
    cv=cv_assigned,
    scoring="f1_macro",
    n_jobs=-1,
)
svm_grid.fit(X_scaled, y)

svm_results = pd.DataFrame(svm_grid.cv_results_)[
    ["param_C", "param_gamma", "mean_test_score", "std_test_score", "rank_test_score"]
]
print(svm_results.sort_values("rank_test_score").to_string(index=False))
print("\nBästa parametrar:", svm_grid.best_params_)

best_svm = SVC(kernel="rbf", **svm_grid.best_params_, random_state=42)

t0 = time.perf_counter()
svm_cv_res = cross_validate(
    best_svm, X_scaled, y, cv=cv_assigned, scoring={"acc": "accuracy", "f1_macro": "f1_macro"}
)
t_svm_cv = time.perf_counter() - t0

t0 = time.perf_counter()
y_pred_svm_loo = cross_val_predict(best_svm, X_scaled, y, cv=LeaveOneOut())
t_svm_loo = time.perf_counter() - t0

print(
    f"\n{n_folds}-fold Stratified CV -> Acc: {svm_cv_res['test_acc'].mean():.4f}, "
    f"F1 macro: {svm_cv_res['test_f1_macro'].mean():.4f}, Tid: {t_svm_cv:.4f}s"
)
print(
    f"LOO CV                 -> Acc: {accuracy_score(y, y_pred_svm_loo):.4f}, "
    f"F1 macro: {f1_score(y, y_pred_svm_loo, average='macro'):.4f}, Tid: {t_svm_loo:.4f}s"
)

 param_C param_gamma  mean_test_score  std_test_score  rank_test_score
     1.0        0.01         1.000000        0.000000                1
    10.0        0.01         1.000000        0.000000                1
   100.0        0.01         1.000000        0.000000                1
     0.1       scale         0.995829        0.008341                4
     1.0         0.1         0.995829        0.008341                4
     1.0       scale         0.995829        0.008341                4
     0.1         0.1         0.995829        0.008341                4
     0.1        0.01         0.995829        0.008341                4
   100.0         0.1         0.995829        0.008341                4
   100.0       scale         0.995829        0.008341                4
    10.0         0.1         0.995829        0.008341                4
    10.0       scale         0.995829        0.008341                4
    10.0           1         0.987488        0.010216               13
     1